# GUI Application

> Tkinter-based control panel for GzKeyboard

In [ ]:
#| default_exp gui

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import tkinter as tk
from tkinter import ttk
import sys

from gzkeyboard.core import CharacterStore, MappingsConfig
from gzkeyboard.system import SystemInputHandler, InputMode, KeyboardConfig, start_keyboard, stop_keyboard

## GUI Application

A small control panel window. The keyboard hook runs in the background while
tkinter's `mainloop()` keeps the app alive — no extra threads needed.

In [ ]:
#| export
class GzKeyboardApp:
    """Tkinter control panel for GzKeyboard.

    Provides Start/Stop, Pause, mode selection, and Settings controls.
    The keyboard hook runs in the background; tkinter mainloop keeps the app alive.
    """

    BG         = '#2b2b2b'
    FG         = '#ffffff'
    BTN_BG     = '#3c3f41'
    BTN_ACTIVE = '#4c5052'
    GREEN      = '#4caf50'
    YELLOW     = '#ffc107'
    RED        = '#f44336'
    ACCENT     = '#4fc3f7'

    MODES = [
        ('Tigrinya', 'tigrinya'),
        ('Amharic',  'amharic'),
        ('Latin',    'latin'),
    ]

    def __init__(self, root: tk.Tk):
        self.root = root
        self.handler: SystemInputHandler = None
        self._settings_win = None
        self._build_ui()
        self._update_ui()

    # --- UI construction ---

    def _build_ui(self):
        self.root.title('GzKeyboard')
        self.root.resizable(False, False)
        self.root.configure(bg=self.BG)
        self.root.protocol('WM_DELETE_WINDOW', self._on_close)

        pad = dict(padx=12, pady=6)

        # Status bar
        self.status_var = tk.StringVar(value='Stopped')
        self.status_label = tk.Label(
            self.root, textvariable=self.status_var,
            bg=self.BG, fg=self.FG,
            font=('Segoe UI', 11, 'bold'), anchor='w'
        )
        self.status_label.grid(row=0, column=0, columnspan=4, sticky='we', padx=12, pady=(12, 4))

        # Mode selector
        mode_frame = tk.LabelFrame(
            self.root, text='Mode',
            bg=self.BG, fg=self.FG,
            font=('Segoe UI', 9)
        )
        mode_frame.grid(row=1, column=0, columnspan=4, sticky='we', padx=12, pady=6)

        self.mode_var = tk.StringVar(value='tigrinya')
        for i, (label, value) in enumerate(self.MODES):
            rb = tk.Radiobutton(
                mode_frame, text=label, variable=self.mode_var, value=value,
                command=self._on_mode_change,
                bg=self.BG, fg=self.FG, selectcolor=self.BTN_BG,
                activebackground=self.BG, activeforeground=self.ACCENT,
                font=('Segoe UI', 10)
            )
            rb.grid(row=0, column=i, padx=10, pady=4)

        # Buttons row
        self.start_btn = self._make_button('Start', self._on_start_stop, self.GREEN)
        self.start_btn.grid(row=2, column=0, **pad)

        self.pause_btn = self._make_button('Pause', self._on_pause, self.YELLOW)
        self.pause_btn.grid(row=2, column=1, **pad)

        settings_btn = self._make_button('Settings', self._on_settings, self.ACCENT)
        settings_btn.grid(row=2, column=2, **pad)

        quit_btn = self._make_button('Quit', self._on_close, self.RED)
        quit_btn.grid(row=2, column=3, **pad)

        for i in range(4):
            self.root.columnconfigure(i, weight=1)

    def _make_button(self, text, command, color):
        return tk.Button(
            self.root, text=text, command=command,
            bg=self.BTN_BG, fg=color, activebackground=self.BTN_ACTIVE,
            activeforeground=color, relief='flat',
            font=('Segoe UI', 10, 'bold'), width=8, pady=4
        )

    # --- Button handlers ---

    def _on_start_stop(self):
        if self.handler is None:
            self.handler = start_keyboard(self.mode_var.get())
            self.paused = False
        else:
            stop_keyboard(self.handler)
            self.handler = None
            self.paused = False
        self._update_ui()

    def _on_pause(self):
        if self.handler is None:
            return
        self.paused = not self.paused
        self.handler.active = not self.paused
        self._update_ui()

    def _on_mode_change(self):
        if self.handler is not None:
            self.handler.switch_mode(InputMode(self.mode_var.get()))
        self._update_ui()

    def _on_settings(self):
        """Open (or focus) the Settings window."""
        # If already open, just bring it to front
        if self._settings_win is not None:
            try:
                self._settings_win.win.lift()
                return
            except tk.TclError:
                self._settings_win = None  # window was closed

        # Need a live handler to read current mappings; if not running, create a
        # temporary one just to get the mappings (no keyboard hook started)
        if self.handler is not None:
            target_handler = self.handler
        else:
            target_handler = SystemInputHandler()

        self._settings_win = SettingsWindow(
            self.root, target_handler,
            bg=self.BG, fg=self.FG, btn_bg=self.BTN_BG, accent=self.ACCENT
        )

        def _on_settings_close():
            self._settings_win = None
            self._settings_win_ref = None

        self._settings_win.win.protocol('WM_DELETE_WINDOW',
                                        lambda: (self._settings_win.win.destroy(),
                                                 setattr(self, '_settings_win', None)))

    def _on_close(self):
        if self.handler is not None:
            stop_keyboard(self.handler)
        self.root.destroy()

    # --- UI state sync ---

    def _update_ui(self):
        running = self.handler is not None

        if not running:
            self.status_var.set('Stopped')
            self.status_label.config(fg=self.RED)
            self.start_btn.config(text='Start', fg=self.GREEN)
            self.pause_btn.config(state='disabled')
        elif self.paused:
            self.status_var.set('Paused')
            self.status_label.config(fg=self.YELLOW)
            self.start_btn.config(text='Stop', fg=self.RED)
            self.pause_btn.config(text='Resume', state='normal', fg=self.GREEN)
        else:
            mode_label = next(l for l, v in self.MODES if v == self.mode_var.get())
            self.status_var.set(f'Running  —  {mode_label}')
            self.status_label.config(fg=self.GREEN)
            self.start_btn.config(text='Stop', fg=self.RED)
            self.pause_btn.config(text='Pause', state='normal', fg=self.YELLOW)

## Entry Point

## Settings Window

A scrollable table that lets the user view and edit key sequence → character family mappings.

In [ ]:
#| export
class SettingsWindow:
    """Settings window for editing key sequence → character family mappings.

    Displays a scrollable table of every sequence in the active MappingsConfig.
    Users can edit the *Sequence* column to remap a family to a different key
    sequence, then Save or Revert to Defaults.

    Columns:
        Sequence  — the Latin key sequence the user types  (editable)
        Family    — the internal family base_key            (read-only)
        Characters — all 7 forms of the family              (read-only)
    """

    def __init__(self, parent: tk.Tk, handler: SystemInputHandler,
                 bg: str, fg: str, btn_bg: str, accent: str):
        self.handler = handler
        self._bg, self._fg = bg, fg
        self._btn_bg, self._accent = btn_bg, accent

        self.win = tk.Toplevel(parent)
        self.win.title('GzKeyboard — Settings')
        self.win.configure(bg=bg)
        self.win.resizable(True, True)
        self.win.minsize(520, 380)

        self._build_ui()
        self._load_mappings(handler.mappings)

    # --- Build UI ---

    def _build_ui(self):
        # Info label
        tk.Label(
            self.win,
            text='Edit the Sequence column to change key mappings. Save to apply.',
            bg=self._bg, fg=self._fg, font=('Segoe UI', 9),
            anchor='w', wraplength=500
        ).pack(fill='x', padx=12, pady=(10, 2))

        # Scrollable table frame
        frame = tk.Frame(self.win, bg=self._bg)
        frame.pack(fill='both', expand=True, padx=12, pady=4)

        cols = ('sequence', 'family', 'characters')
        self._tree = ttk.Treeview(frame, columns=cols, show='headings', height=18)
        self._tree.heading('sequence',   text='Sequence')
        self._tree.heading('family',     text='Family')
        self._tree.heading('characters', text='Characters (all forms)')
        self._tree.column('sequence',   width=90,  anchor='center')
        self._tree.column('family',     width=90,  anchor='center')
        self._tree.column('characters', width=300, anchor='w')

        vsb = ttk.Scrollbar(frame, orient='vertical',   command=self._tree.yview)
        hsb = ttk.Scrollbar(frame, orient='horizontal', command=self._tree.xview)
        self._tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)

        self._tree.grid(row=0, column=0, sticky='nsew')
        vsb.grid(row=0, column=1, sticky='ns')
        hsb.grid(row=1, column=0, sticky='ew')
        frame.rowconfigure(0, weight=1)
        frame.columnconfigure(0, weight=1)

        # Bind double-click to edit sequence
        self._tree.bind('<Double-1>', self._on_double_click)

        # Buttons
        btn_frame = tk.Frame(self.win, bg=self._bg)
        btn_frame.pack(fill='x', padx=12, pady=(4, 10))

        tk.Button(
            btn_frame, text='Save', command=self._on_save,
            bg=self._btn_bg, fg='#4caf50', activebackground='#4c5052',
            relief='flat', font=('Segoe UI', 10, 'bold'), width=10
        ).pack(side='left', padx=(0, 6))

        tk.Button(
            btn_frame, text='Revert to Defaults', command=self._on_revert,
            bg=self._btn_bg, fg='#ffc107', activebackground='#4c5052',
            relief='flat', font=('Segoe UI', 10, 'bold')
        ).pack(side='left')

        self._status_var = tk.StringVar(value='')
        tk.Label(btn_frame, textvariable=self._status_var,
                 bg=self._bg, fg=self._accent,
                 font=('Segoe UI', 9)).pack(side='right')

    # --- Data loading ---

    def _load_mappings(self, mappings: MappingsConfig):
        """Populate the treeview from a MappingsConfig."""
        store = CharacterStore()
        # Build reverse map: family → sequence (using current mappings)
        family_to_seq = {fam: seq for seq, fam in mappings.sequence_to_family.items()}

        # Clear existing rows
        for row in self._tree.get_children():
            self._tree.delete(row)

        # Sort: single-char sequences first, then multi-char alphabetically
        def sort_key(item):
            seq, fam = item
            return (len(seq), seq)

        for seq, fam in sorted(mappings.sequence_to_family.items(), key=sort_key):
            family_obj = store.get_family(fam)
            if family_obj:
                chars = ' '.join(c for c in family_obj.characters if c is not None)
            else:
                chars = '(unknown family)'
            self._tree.insert('', 'end', iid=fam, values=(seq, fam, chars))

        # Store editable sequences in a dict for diffing on save
        self._edited: dict = dict(mappings.sequence_to_family)

    # --- Inline editing ---

    def _on_double_click(self, event):
        """Open an inline entry widget over the Sequence cell when double-clicked."""
        region = self._tree.identify_region(event.x, event.y)
        if region != 'cell':
            return
        col = self._tree.identify_column(event.x)
        if col != '#1':   # only the Sequence column is editable
            return
        row_id = self._tree.identify_row(event.y)
        if not row_id:
            return

        # Get cell bounding box
        x, y, w, h = self._tree.bbox(row_id, col)
        current_seq = self._tree.set(row_id, 'sequence')

        entry_var = tk.StringVar(value=current_seq)
        entry = tk.Entry(self._tree, textvariable=entry_var,
                         font=('Segoe UI', 10), width=12)
        entry.place(x=x, y=y, width=w, height=h)
        entry.focus_set()
        entry.select_range(0, 'end')

        def commit(e=None):
            new_seq = entry_var.get().strip().lower()
            if new_seq:
                self._tree.set(row_id, 'sequence', new_seq)
                self._edited[row_id] = new_seq  # row_id == family base_key
            entry.destroy()

        entry.bind('<Return>',    commit)
        entry.bind('<FocusOut>',  commit)
        entry.bind('<Escape>',    lambda e: entry.destroy())

    # --- Button handlers ---

    def _on_save(self):
        """Build a new MappingsConfig from current table values and apply it."""
        # Read current values from treeview (respects any inline edits)
        new_seq_to_fam: dict = {}
        for row_id in self._tree.get_children():
            seq = self._tree.set(row_id, 'sequence').strip().lower()
            fam = self._tree.set(row_id, 'family')
            if seq:
                new_seq_to_fam[seq] = fam

        new_mappings = MappingsConfig(sequence_to_family=new_seq_to_fam)

        if self.handler is not None:
            self.handler.apply_mappings(new_mappings)

        self._status_var.set('Saved.')
        self.win.after(2000, lambda: self._status_var.set(''))

    def _on_revert(self):
        """Reset to factory defaults and reload the table."""
        defaults = MappingsConfig.default()
        self._load_mappings(defaults)
        # Also apply immediately if keyboard is running
        if self.handler is not None:
            self.handler.apply_mappings(defaults)
        self._status_var.set('Reverted to defaults.')
        self.win.after(2000, lambda: self._status_var.set(''))

In [ ]:
#| export
def main():
    """Launch the GzKeyboard GUI control panel."""
    root = tk.Tk()
    app = GzKeyboardApp(root)
    root.mainloop()

if __name__ == '__main__':
    main()

## Tests

In [ ]:
#| eval: false
# Smoke test: build the UI and verify initial state (no window displayed)
root = tk.Tk()
root.withdraw()
app = GzKeyboardApp(root)

test_eq(app.handler, None)
test_eq(app.paused, False)
test_eq(app.mode_var.get(), 'tigrinya')
test_eq(app.status_var.get(), 'Stopped')
test_eq(str(app.pause_btn['state']), 'disabled')

root.destroy()
print('GUI smoke test passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()